# Banc d'essai `bench` — annexes

Annexes de `notebook_generation_bench.ipynb`, déplacées ici **à l'identique**
lors de l'amincissement du notebook principal (septembre 2026) : deux
générations (A, non utilisé), `prompt_local.py` (B), itération par copie d'un
dossier scénario (C), ajout de DAS sur un scénario (D). Exécuter d'abord la
cellule de contexte ci-dessous (mêmes paramètres et mêmes noms que le
notebook principal : `TD`, `MODEL`, `PRICING`, `mistral_client`, `show_crh`…),
puis les cellules de l'annexe voulue. Les cellules « run réel » exigent
`MISTRAL_API_KEY` dans l'environnement du noyau.

In [ ]:
# --- Contexte : paramètres du notebook principal (à garder alignés) ---
import importlib.util
import json
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

REPO_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                  if (p / "bench").is_dir() and (p / "core").is_dir()), None)
assert REPO_ROOT, "Racine du repo Stream introuvable — lancer le notebook depuis generation/."
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import polars as pl
from IPython.display import display
from bench import (
    BenchError, Pricing, copy_system_prompts, generate, load_reports,
    scenario_dirs, seed_user_prompts, user_from_column, write_prompts,
)
from bench.banc import *

TEST_NUM = "06"
PREV_TEST = "05"
SOURCE_PROFILES_PATH = REPO_ROOT / "data/aphp/scenarios_bn_all_20260128.pq"
TD, PREV_TD = dossiers_test(TEST_NUM, PREV_TEST)

MODEL = os.environ.get("MISTRAL_MODEL", "mistral-large-latest")
MAX_TOKENS_SUMMARY = 8_000
MAX_TOKENS_CR = 128_000
TRANSPORT = os.environ.get("MISTRAL_TRANSPORT", "sync")
MAX_WORKERS = 3
_TARIFS = {"sync": (0.5, 1.5), "batch": (0.25, 0.75)}
PRICING = Pricing(*_TARIFS[TRANSPORT])
SYSTEM_PROMPT_FILE = "prompt_system_one_gen.txt"
OUT_FILE = "crh_generation.txt"

# Prefill de la première génération d'un test deux temps (annexe A, non
# utilisé — repris de l'ancien notebook).
FIRST_GEN_PREFIX = "Résumé clinique :"

# Injection du résumé intermédiaire dans le user prompt du second temps
# (annexe A, §4).
SUMMARY_HEADER = """### RÉSUMÉ CLINIQUE ISSU DE LA PREMIÈRE GÉNÉRATION
Le résumé ci-dessous est une aide intermédiaire.
Le scénario clinique, les codes, les fiches descriptives et les instructions
restent prioritaires en cas de divergence."""

SUMMARY_FOOTER = "### FIN DU RÉSUMÉ CLINIQUE INTERMÉDIAIRE"


def show_crh(scenario: str, out: str | None = None) -> None:
    """Affiche en markdown le CR d'un dossier scénario du test courant."""
    afficher_crh(TD, scenario, out or OUT_FILE)


print("Test courant :", TD, "(existe)" if TD.is_dir() else "(à créer)")
print("Transport :", TRANSPORT, "— tarifs ($ / 1M tokens) :", PRICING)

## Annexe A — deux générations (non utilisé)

Le workflow 2-gen (résumé puis CR : deux `generate`, le `reports` du premier
nourrissant le `context` du second, spec §4) n'est **pas utilisé** — conservé
pour le jour venu. Positions : `first_gen` et `second_gen`, puis chaîne comme
le workflow principal. Les jeux d'amorçage `template_first_gen` /
`template_second_gen` ont été supprimés avec l'ancien monde : ils restent
dans l'historique git (dernier commit les contenant : `339b2b4`). Cellules
volontairement non exécutables — copier dans des cellules code pour les
activer.

```python
# Graine (test dédié) puis montage des DEUX positions
seed_user_prompts(TD, selected_scenarios, seed_path=SOURCE_PROFILES_PATH)
# jeux d'amorçage : dans l'historique git (commit 339b2b4) — les restaurer
# puis les copier dans TD/system/first_gen et TD/system/second_gen, ex. :
#   git checkout 339b2b4 -- work_modif_prompts/template_first_gen  # puis copie

# [édition des jeux dans TD/system/first_gen/ et TD/system/second_gen/]
print("Figé first_gen  :", copy_system_prompts(TD, "first_gen"))
print("Figé second_gen :", copy_system_prompts(TD, "second_gen"))

# Premier temps — contrôle à sec (complétude) puis run réel
dry2a = generate(TD, system="prompt_system_first_gen.txt",
                 user="user_generation.txt", out="crh_resume.txt",
                 client=None, model=MODEL, max_tokens=MAX_TOKENS_SUMMARY,
                 pricing=PRICING, prefix_text=FIRST_GEN_PREFIX, dry_run=True)
show_first_prompt(dry2a)

res1 = generate(TD, system="prompt_system_first_gen.txt",
                user="user_generation.txt", out="crh_resume.txt",
                client=mistral_client(), model=MODEL,
                max_tokens=MAX_TOKENS_SUMMARY, pricing=PRICING,
                prefix_text=FIRST_GEN_PREFIX)

# Second temps — le résumé nourrit le contexte (à sec : placeholder possible,
# même geste que la reprise du §8)
cr2 = generate(TD, system="prompt_system_second_gen.txt",
               user="user_generation.txt", out="crh_final.txt",
               client=mistral_client(), model=MODEL,
               max_tokens=MAX_TOKENS_CR, pricing=PRICING,
               prefix_file="prefix.txt",  # prefill d'origine de la graine
               context=res1.reports,      # ou load_reports(TD, "crh_resume.txt")
               context_header=SUMMARY_HEADER, context_footer=SUMMARY_FOOTER)
print(cr2.usage)
```


## Annexe B — `prompt_local.py` — logique de user prompt locale au test (§3.7)

Pour tester une **construction** de user prompt différente sans toucher au
package : un `prompt_local.py` à la racine du test (c'est un fichier : la
découverte l'ignore), chargé ici et passé en `user_fn=` à `seed_user_prompts`
d'un **nouveau** test — changer `TEST_NUM`/`PREV_TEST` en tête de notebook,
puis utiliser la graine ci-dessous **à la place** de celle de la section 3 ;
la suite (montage, figement, dry-run) est le workflow principal, inchangé.
Hiérarchie des leviers : (1) éditer les `.txt` du test ; (2) `prompt_local.py` ;
(3) monkeypatch fictomed (fragile, à noter dans `test.json["notes"]`) ;
(4) modifier le clone fictomed éditable.


In [ ]:
TD.mkdir(parents=True, exist_ok=True)

_prompt_local_path = TD / "prompt_local.py"
if not _prompt_local_path.exists():  # ne jamais écraser une version éditée
    _prompt_local_path.write_text(
        '''"""User prompt local au test (spec §3.7) — exemple.

Point de départ possible : inspect.getsource sur la fonction fictomed
correspondante, copiée puis modifiée.
"""


def build_user(row: dict) -> str:
    """User prompt fictomed + rappel explicite du DP et du GHM."""
    return (
        str(row["user_prompt"]).rstrip()
        + "\\n\\nRappel codage : DP "
        + str(row.get("icd_primary_code"))
        + " — GHM "
        + str(row.get("ghm2"))
        + "\\n"
    )
''',
        encoding="utf-8",
    )
    print("Écrit :", _prompt_local_path)

import importlib.util

_spec_local = importlib.util.spec_from_file_location(
    f"prompt_local_{TEST_NUM}", _prompt_local_path
)
prompt_local = importlib.util.module_from_spec(_spec_local)
_prev_dwb = sys.dont_write_bytecode
sys.dont_write_bytecode = True  # pas de __pycache__ dans le dossier de test
try:
    _spec_local.loader.exec_module(prompt_local)
finally:
    sys.dont_write_bytecode = _prev_dwb

print("Chargé :", prompt_local.build_user.__doc__)

In [ ]:
# Graine avec user_fn — À LA PLACE de la graine de la section 3.1, sur un test
# neuf. Refus normal (BenchError) si le test courant est déjà seedé.
if "selected_scenarios" not in globals():
    print("SKIP — pas de scénarios en mémoire (la chaîne fictomed n'a pas "
          "tourné : test déjà seedé).")
else:
    try:
        print(
            "Scénarios créés :",
            seed_user_prompts(
                TD,
                selected_scenarios,
                user_fn=prompt_local.build_user,
                seed_path=SOURCE_PROFILES_PATH,
            ),
        )
    except BenchError as exc:
        print("Graine refusée (test déjà seedé — normal en re-run) :", exc)

## Annexe C — itérer par copie d'un dossier scénario

Un dossier scénario est autonome : sa copie emporte **tous** ses prompts (et ses
sorties éventuelles). Nom libre ; le prochain `generate` sur le test l'inclut
dans la découverte — `only=` permet de ne relancer que la copie.


In [ ]:
_names = scenario_dirs(TD)
if not _names:
    print("Pas de dossiers scénario — seeder d'abord (3.1) ; copie skippée.")
else:
    _src = TD / _names[0]           # n'importe quel scénario servi
    _dst = TD / f"{_src.name}_bis"  # nom libre
    if _dst.exists():
        print("Copie déjà présente :", _dst)
    else:
        shutil.copytree(_src, _dst)
        print("Copié :", _src.name, "->", _dst.name)
    print("Découverte :", scenario_dirs(TD))

In [ ]:
# contrôle à sec — vérifie le prompt assemblé de la copie
if "_dst" not in globals() or not _dst.exists():
    print("Pas de copie de scénario — cellule précédente skippée.")
else:
    dry_copie = generate(
        TD,
        system="prompt_system_one_gen.txt",
        user="user_generation.txt",
        out="crh_generation.txt",
        client=None,  # inutile en dry-run
        model=MODEL,
        max_tokens=MAX_TOKENS_CR,
        pricing=PRICING,
        prefix_file="prefix.txt",
        only=[_dst.name],
        dry_run=True,
    )
    print(dry_copie.reports["system_prompt"][0][-1500:])

In [ ]:
# run réel, une seule requête — les autres dossiers restent intacts
cr_copie = generate(
    TD,
    system="prompt_system_one_gen.txt",
    user="user_generation.txt",
    out="crh_generation.txt",
    client=mistral_client(),  # la clé vient de l'environnement
    transport=TRANSPORT,
    max_workers=MAX_WORKERS,
    model=MODEL,
    max_tokens=MAX_TOKENS_CR,
    pricing=PRICING,
    prefix_file="prefix.txt",
    only=[_dst.name],
)
print(cr_copie.usage)

In [ ]:
if "_dst" in globals() and _dst.exists():
    show_crh(_dst.name)  # lecture du CR régénéré de la copie
else:
    print("Pas de copie de scénario — rien à lire.")

## Annexe D — tester l'ajout de DAS sur un scénario

`scenario_bis_avec_das("0002", ["I10", "E119"], suffix="t1")` copie le dossier
`0002` en `0002_t1` et ajoute les codes à la liste « Diagnostics associés » du
user prompt **avec leurs fiches descriptives** (référentiels fictomed :
fiche exacte, sinon fiche de catégorie ; codes déjà présents ou invalides
ignorés ; sorties purgées de la copie ; figement/template/prefix de la base
conservés). Le scénario de base reste intact — comparaison directe base/bis.
Puis `regen_crh("0002_t1")` régénère **ce seul CRH** (run réel : clé requise)
et écrit son `.md` à côté du `.txt`.


In [ ]:
import re

In [ ]:
# Test d'ajout de DAS : copie un dossier scénario en <base>_<suffix>, ajoute
# les codes à la liste « Diagnostics associés » du user prompt AVEC leurs
# fiches descriptives (référentiels fictomed), puis régénère ce seul CRH.
# Le scénario de base reste intact — la comparaison base/bis est directe.
from fictomed.sites.aphp.code_cards import CodeCardsRegistry, normalize_icd_code

_CARDS = CodeCardsRegistry.from_dirs(
    exact_dir=REPO_ROOT / "data/aphp/referentials/cards_library",
    category_dir=REPO_ROOT / "data/aphp/referentials/cards_library_categories",
)


def _libelle(card_text: str, code: str) -> str:
    m = re.search(r"^# \S+ — (.+)$", card_text, re.M)
    return m.group(1).strip() if m else f"Diagnostic associé ({code})"


def scenario_bis_avec_das(base: str, codes_das: list[str], *, suffix: str = "t1") -> str:
    """Crée le dossier `<base>_<suffix>` = scénario `base` + DAS ajoutés.

    - lignes ajoutées à « * Diagnostics associés : » (libellé + code) ;
    - fiches des nouveaux codes ajoutées à la fin (déjà présentes : skip) ;
    - sorties (crh, .md, verdict) purgées de la copie ;
    - figement, template, prefix : ceux de la base (copiés tels quels).
    """
    src, dst = TD / base, TD / f"{base}_{suffix}"
    if not src.is_dir():
        raise BenchError(f"Dossier scénario absent : {src}")
    if dst.exists():
        raise BenchError(f"{dst} existe déjà — choisir un autre suffix.")
    shutil.copytree(src, dst)
    for stale in dst.glob("crh_*"):
        stale.unlink()
    (dst / "verdict.txt").unlink(missing_ok=True)

    user_path = dst / "user_generation.txt"
    prompt = user_path.read_text(encoding="utf-8")

    lignes, fiches, sans_fiche = [], [], []
    for raw in codes_das:
        code = normalize_icd_code(raw)
        if not code:
            print(f"  {raw!r} : code invalide — ignoré")
            continue
        code_liste = code.replace(".", "")  # convention du scénario : sans point
        if f"({code_liste})" in prompt or f"({code})" in prompt:
            print(f"  {code_liste} : déjà dans le scénario — ignoré")
            continue
        card = _CARDS.find(code)
        if card is None:
            sans_fiche.append(code_liste)
            lignes.append(f"Diagnostic associé ajouté ({code_liste})")
            continue
        lignes.append(f"{_libelle(card.text, code)} ({code_liste})")
        if f'code="{card.card_code}"' not in prompt:
            fiches.append(card.text.strip())
    if not lignes:
        shutil.rmtree(dst)
        raise BenchError("Aucun code à ajouter — dossier bis non créé.")

    # 1. lignes DAS — sous « * Diagnostics associés : » (créée au besoin)
    m = re.search(r"^(\s*)\* Diagnostics associés :\s*$", prompt, re.M)
    if m:
        indent = m.group(1) + "   - "
        insertion = "".join(f"\n{indent}{l}" for l in lignes)
        prompt = prompt[: m.end()] + insertion + prompt[m.end():]
    else:
        m = re.search(r"^(\s*)\* Diagnostic principal :.*$", prompt, re.M)
        if not m:
            shutil.rmtree(dst)
            raise BenchError("Ancre « Diagnostic principal » introuvable dans le user prompt.")
        indent = m.group(1)
        bloc = f"\n{indent}* Diagnostics associés :" + "".join(
            f"\n{indent}   - {l}" for l in lignes
        )
        prompt = prompt[: m.end()] + bloc + prompt[m.end():]

    # 2. fiches des nouveaux codes — après la dernière fiche existante
    # (les fiches exactes ferment par </fiche_code>, les fiches de
    #  catégorie par </fiche_category>)
    if fiches:
        bloc = "\n\n" + "\n\n".join(fiches)
        fin, tag = max(
            (prompt.rfind(t), t) for t in ("</fiche_code>", "</fiche_category>")
        )
        if fin >= 0:
            fin += len(tag)
            prompt = prompt[:fin] + bloc + prompt[fin:]
        else:
            prompt = prompt.rstrip() + bloc + "\n"

    user_path.write_text(prompt, encoding="utf-8")
    print(f"Créé : {dst.name} — {len(lignes)} DAS ajouté(s), "
          f"{len(fiches)} fiche(s) insérée(s)"
          + (f", SANS fiche : {sans_fiche}" if sans_fiche else ""))
    return dst.name


def regen_crh(name: str, *, out: str | None = None) -> None:
    """Run réel du seul scénario `name`, puis .md à côté du .txt + lecture inline."""
    out = out or OUT_FILE
    cr = generate(
        TD,
        system=SYSTEM_PROMPT_FILE,
        user="user_generation.txt",
        out=out,
        client=mistral_client(),
        model=MODEL,
        max_tokens=MAX_TOKENS_CR,
        pricing=PRICING,
        prefix_file="prefix.txt",
        only=[name],
    )
    print(cr.usage)
    import subprocess

    subprocess.run(
        [sys.executable, str(REPO_ROOT / "scripts" / "show_crh.py"),
         str(TD / name), "--md", "--out", out],
        check=False,
    )
    show_crh(name, out=out)


E6603 Surpoids dû à un excès calorique, de l’adulte ou de l’enfant
E6604 Obésité due à un excès calorique de l’adulte avec indice de masse corporelle [IMC] égal ou supérieur à 30 kg/m² et inférieur à 35 kg m², ou obésité due à un excès calorique de l’enfant
E6605 Obésité due à un excès calorique de l’adulte avec indice de masse corporelle [IMC] égal ou supérieur à 35 kg/m² et inférieur à 40 kg/m²
E6606 Obésité due à un excès calorique de l’adulte avec indice de masse corporelle [IMC] égal ou supérieur à 40 kg/m² et inférieur à 50 kg/m²
E6607 Obésité due à un excès calorique de l’adulte avec indice de masse corporelle [IMC] égal ou supérieur à 50 kg/m²


F1720	Syndrome de dépendance au tabac, personne actuellement abstinente
F1724	Syndrome de dépendance au tabac, utilisation actuelle


F1020	Syndrome de dépendance à l'alcool, personne actuellement abstinente
F1024	Syndrome de dépendance à l'alcool, utilisation actuelle
F1026	Syndrome de dépendance à l'alcool, utilisation épisodique

In [ ]:
# Exemple — à adapter puis décommenter :
nom = scenario_bis_avec_das("0008", ["E6600", "F1720","F1020"], suffix="t1")
regen_crh(nom)